# A4 — Bootstrap CI on Win Rate

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
from strategy import bootstrap_ci

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, (wk, wm) in zip(axes, WINDOWS_META.items()):
    rk = wm['result_key']
    trades = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'trades.parquet')
    win_flag = (trades['gross_usd'] > 0).astype(int).values
    lo, mid, hi = bootstrap_ci(win_flag, stat_fn=np.mean, n_boot=5000, ci=0.95)
    ax.errorbar([0], [mid], yerr=[[mid-lo],[hi-mid]], fmt='o', capsize=8,
                color=WIN_COLORS[wk], markersize=8)
    ax.axhline(0.5, color='black', ls='--', lw=0.8)
    ax.set_title(f'{wk}\nn={len(win_flag)}, WR={mid:.3f}')
    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
axes[0].set_ylabel('Win Rate')
fig.suptitle('Bootstrap 95% CI on Win Rate — All Windows', fontsize=13)
save_fig(fig, 'A4_bootstrap_ci_win_rate.png')
